# 환경 범주 민감도

- 대응 코드: `sensitivity_e_subcategory.py`
- 환경 키워드의 오탐 가능성과 제외 시나리오를 확인.
- 시나리오 계산은 선택 실행.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
RUN_SENSITIVITY = False
PREVIEW_ROWS = 10

In [ ]:
from stage2_frequency import sensitivity_e_subcategory as step
for lang in cfg.LANGS:
    merged, docs = load_enriched(lang)
    e_mentions = merged[merged["category"].isin(step.E_CATS)]
    print(f"{lang}: E 관련 언급 {len(e_mentions):,}건")
    display(e_mentions.groupby("canonical").size().sort_values(ascending=False).head(PREVIEW_ROWS).rename("mentions").to_frame())

In [ ]:
if RUN_SENSITIVITY:
    step.main()
else:
    print("민감도 시나리오 계산은 RUN_SENSITIVITY=True로 선택하세요.")

In [ ]:
show_csv(step.OUT, PREVIEW_ROWS)